# Trustworthy LLM Agents — PyConf Hyderabad 2026

## Observability, Evals & Security Guardrails for LLM Agents

---

# Setting the Context

## LLMs, Agents & Multi-Agent Systems

### Large Language Models (LLMs)

LLMs are foundation models trained on massive text corpora that can understand and generate human language. They power everything from chatbots to code assistants — but on their own, they are **stateless text predictors**. They don't remember, they don't act, and they don't verify.

### Agents

An **agent** is an LLM wrapped with:
- **Tools** — functions the LLM can call (database lookups, APIs, calculations)
- **Memory** — conversation history or retrieved context
- **A decision loop** — the LLM decides which tool to call, processes the result, and decides the next step

This turns a passive text generator into an **autonomous actor** that can reason, plan, and execute multi-step tasks.

### Multi-Agent Systems

In production, a single agent rarely handles everything. Instead, you build a **swarm** of specialised agents:

```
User Query → Router Agent → Specialist Agent(s) → Response
                  │
                  ├── OrderSupport Agent (lookup, refund, email)
                  ├── PolicyAdvisor Agent (knowledge base search)
                  └── EscalationAgent (priority, team assignment)
```

Each agent has its own system prompt, tools, and guardrails. The **router** classifies intent and delegates to the right specialist.

### The trust problem

Agents are powerful — but they introduce new failure modes that traditional software doesn't have:
- The LLM can **hallucinate** tool arguments (refund $500 on a $50 order)
- The LLM can **skip safety steps** when socially engineered by the user
- The LLM can **leak sensitive data** across customer boundaries
- The LLM can **obey malicious instructions** injected through tool output data

None of these produce errors. None of these crash the app. They look like successful responses — and that's what makes them dangerous.

## Why Observability, Evals & Guardrails?

Traditional software has well-established patterns for reliability — unit tests, integration tests, logging, monitoring. But LLM agents break these patterns:

| Traditional Software | LLM Agents |
|---|---|
| Deterministic — same input, same output | **Non-deterministic** — same input, different output |
| Errors are explicit (exceptions, status codes) | **Failures are silent** (wrong answer, no error) |
| Logic is in code (inspectable, testable) | **Logic is in prompts** (opaque, fragile) |
| Input is validated at boundaries | **Input flows through LLM** (injection risk) |
| Access control is enforced by code | **Access control depends on LLM compliance** |

This is why we need three new pillars:

### Observability
You can't fix what you can't see. Observability gives you **X-ray vision** into every LLM call, tool invocation, and agent decision — traces, spans, token usage, latency, cost.

### Evaluations (Evals)
You can't test LLM agents with `assert output == expected`. Instead, you need **evaluation frameworks** that grade agent behaviour on correctness, safety, and policy compliance across diverse test cases.

### Guardrails
You can't trust the LLM to always do the right thing. Guardrails are **runtime safety nets** — input/output validation, tool call interception, PII redaction, and human-in-the-loop checkpoints that enforce rules the LLM might ignore.

## Disclaimer

**This workshop is NOT an Agents 101.**

We assume you already understand the basics of LLMs, prompts, and agent architectures. We won't be teaching you how to build agents from scratch.

Instead, this workshop focuses on what happens **after** you build an agent:
- How do you **observe** what your agent is doing in production?
- How do you **evaluate** whether your agent is behaving correctly?
- How do you **guard** against the failure modes that only appear at runtime?

We'll use a fully functional multi-agent e-commerce support system as our playground — the agents are already built. Your job is to make them **trustworthy**.

---

# Meet the Presenters

### Sanchit Balchandani

**Organization:** EPAM Systems

**Role:** Engineering Manager & Python Practice Lead

Sanchit Balchandani leads the **Python Practice** at EPAM India. With over 16 years of experience, his expertise includes:

- Python backend systems and distributed architectures.
- Applied Generative AI and agent-based systems.

A prominent figure in the Python community, he served as **Chair of PyCon India 2023** and **PyConf Hyderabad 2022**, and is a frequent speaker and workshop lead at industry events.

- **Interests:** Agent-based systems, RAG, Developer productivity with GenAI, Responsible AI.
- **Links:** [Website](https://inovizz.com) | [LinkedIn](https://www.linkedin.com/in/inovizz)

---

### Dineshsuriya D

**Organization:** EPAM Systems

**Role:** Software Engineer

Dineshsuriya D is a Software Engineer at EPAM Systems, specializing in enterprise-scale **Generative AI** solutions. He has hands-on experience building:

- **Agentic workflows** and RAG pipelines.
- AI-powered backend systems focused on robustness, observability, and scalability.

He is an active contributor to open-source GenAI projects, including the **Microsoft Agent Framework** and **Microsoft GraphRAG**, and frequently delivers technical talks on applied GenAI system design.

- **Interests:** Agentic AI systems, RAG, Enterprise GenAI architecture, Backend engineering for AI platforms, Responsible AI.
- **Links:** [GitHub](https://github.com/droideronline) | [LinkedIn](https://www.linkedin.com/in/dinesh106/)

---

# How This Workshop is Structured

This workshop is divided into three main sections, each building on the previous one:

```
┌──────────────────────────────────────────────────────────┐
│  Part 1: Observability                                   │
│  ─────────────────────                                   │
│  Set up Langfuse, understand traces & spans,             │
│  debug 4 silently-failing prompt scenarios               │
│  through the observability dashboard.                    │
├──────────────────────────────────────────────────────────┤
│  Part 2: Evaluations                                     │
│  ────────────────────                                    │
│  Build evaluation datasets, run evals against            │
│  the agent, measure correctness and safety               │
│  across diverse test cases.                              │
├──────────────────────────────────────────────────────────┤
│  Part 3: Guardrails                                      │
│  ───────────────────                                     │
│  Add runtime safety nets — input validation,             │
│  tool call interception, checkpointers,                  │
│  and middleware that enforce rules the LLM               │
│  might ignore.                                           │
└──────────────────────────────────────────────────────────┘
```

### Format

- **Live demo driven** — we present concepts, then immediately demonstrate them on the running app
- **Separate branches** — each section has its own Git branch with the relevant code changes
- **Notebook as guide** — each section has a companion notebook (like this one) that walks through the concepts and demo steps

| Section | Branch | What you'll learn |
|---|---|---|
| Observability | `observability` | Traces, spans, prompt debugging, prompt caching |
| Evaluations | `evaluations` | Eval datasets, metrics, automated grading |
| Guardrails | `guardrails` | Middleware, checkpointers, runtime safety |

---

# The Application: E-Commerce Customer Support

Our demo application is a **multi-agent customer support system** for an online electronics store. Customers interact through a chat interface, and a team of specialised AI agents handles their requests.

### What the app does

A customer sends a message ("I want a refund for ORD-1001") and the system:

1. **Routes** the message — a Router agent classifies the intent (order support, policy question, escalation, or general)
2. **Delegates** to a specialist — the appropriate agent takes over with its own tools and rules
3. **Executes** multi-step workflows — the agent calls tools (database lookups, refund processing, email sending) in a loop until the task is complete
4. **Responds** to the customer — a natural language summary of what was done

### The agents

| Agent | Role | Tools |
|---|---|---|
| **Router** | Classifies customer intent and delegates | None (pure LLM classification) |
| **ShopAssist** | Handles order lookups, refunds, and emails | `lookup_order`, `process_refund`, `send_email`, `search_knowledge_base` |
| **PolicyAdvisor** | Answers policy questions from the knowledge base | `search_knowledge_base` |
| **EscalationAgent** | Handles upset customers, priority assignment, team routing | `search_knowledge_base`, `send_email` |

### The data

The database is seeded with:
- **4 customers** — Alice, Bob, Charlie, Eve
- **7 orders** — various statuses (delivered, shipped, pending), one with a prompt injection payload in the notes field
- **5 knowledge articles** — return policy, refund policy, shipping policy, warranty terms, escalation guidelines (with vector embeddings for semantic search)

## Tech Stack — What We Use and Why

| Technology | What it does | Why we chose it |
|---|---|---|
| **LangGraph** | Orchestrates the multi-agent workflow as a state graph | First-class support for multi-agent patterns, built-in state management, and native tool calling |
| **LangChain** | Provides the LLM abstractions, tool framework, and agent primitives | Industry-standard framework with broad model support (OpenAI, Azure, Anthropic) and a rich tool ecosystem |
| **OpenAI GPT-4o-mini** | The LLM powering all agent reasoning | Fast, cheap ($0.15/1M input tokens), and capable enough for tool-calling workflows |
| **PostgreSQL 16 + pgvector** | Application database with vector search | Single database for both relational data (orders, customers) and semantic search (knowledge articles with embeddings) |
| **SQLAlchemy** | Python ORM for database operations | Type-safe models, session management, and native pgvector support |
| **Pydantic** | Data validation and settings management | Validates tool inputs, agent configs, and environment settings with clear error messages |
| **Docker Compose** | Runs the full stack locally | One command to start all services — no manual setup |
| **Langfuse** (self-hosted) | Observability — traces, spans, token usage, cost | Open-source, self-hosted (system prompts never leave your machine), OpenTelemetry-native |
| **Jinja2** | Template engine for agent prompts | Declarative YAML agents with dynamic variable injection (store name, customer message, order ID) |

### Architecture overview

```
┌─────────────┐     ┌──────────────────────────────────────────────┐
│   Chat UI   │────▶│           LangGraph Dev Server               │
│  (Next.js)  │◀────│                                              │
│  :3000      │     │  ┌────────┐   ┌───────────┐   ┌───────────┐ │
└─────────────┘     │  │ Router │──▶│ShopAssist │   │ PolicyAdv.│ │
                    │  └────────┘   └─────┬─────┘   └───────────┘ │
                    │       │             │                        │
                    │       │       ┌─────┴──────┐                 │
                    │       │       │   Tools    │                 │
                    │       │       │ (DB, Email)│                 │
                    │       │       └─────┬──────┘                 │
                    │       ▼             ▼          :2024         │
                    └──────────────────────────────────────────────┘
                                          │
                    ┌─────────────────┐   │   ┌──────────────────┐
                    │  PostgreSQL 16  │◀──┘──▶│    Langfuse      │
                    │  + pgvector     │       │  (self-hosted)   │
                    │  :5432          │       │  :4000           │
                    └─────────────────┘       └──────────────────┘
```

---

# Part 1: Observability

In this section, we will:
- Set up the observability stack (Langfuse via Docker)
- Understand traces and spans — the building blocks of observability
- Explore why system prompts are sensitive and shouldn't be freely traced
- Walk through 4 real prompt failure scenarios that produce no errors but are silently wrong
- Demonstrate prompt caching and how to observe cache hits/misses

**Switch to the `observability` branch and open its companion notebook to begin.**

```bash
git checkout observability
```

---

# Part 2: Evaluations

In this section, we will:
- Understand why traditional testing doesn't work for LLM agents
- Build evaluation datasets with diverse test cases
- Run automated evals that grade agent behaviour on correctness and safety
- Measure the impact of prompt changes and guardrails on eval scores

**Switch to the `evaluations` branch and open its companion notebook to begin.**

```bash
git checkout evaluations
```

---

# Part 3: Guardrails

In this section, we will:
- Add middleware that intercepts and validates tool calls before execution
- Implement checkpointers for human-in-the-loop approval workflows
- Add input/output guardrails that enforce rules the LLM might ignore
- Build defence-in-depth: combine prompt guardrails with server-side validation

**Switch to the `guardrails` branch and open its companion notebook to begin.**

```bash
git checkout guardrails
```

---

*Workshop by Dineshsuriya D & Sanchit Balchandani — PyConf Hyderabad 2026*